In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

In [2]:
!pip install wandb -q

import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/sandro/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
%run data_extraction_feature_engineering.ipynb
df_train_full, df_test_full = extract_data()
VAL_START_DATE = '2012-09-01'

Loading data...
Train shape: (421570, 5)
Features shape: (8190, 12)
Stores shape: (45, 3)
Test shape: (115064, 4)
Merging datasets...

Data Preparation Complete!
X_train shape: (397841, 24)
X_val shape: (23729, 24)
   IsHoliday  Temperature  Fuel_Price  MarkDown1  MarkDown2  MarkDown3  \
0      False        42.31       2.572        0.0        0.0        0.0   
1       True        38.51       2.548        0.0        0.0        0.0   
2      False        39.93       2.514        0.0        0.0        0.0   
3      False        46.63       2.561        0.0        0.0        0.0   
4      False        46.50       2.625        0.0        0.0        0.0   

   MarkDown4  MarkDown5         CPI  Unemployment  ...  sales_lag_52  \
0        0.0        0.0  211.096358         8.106  ...           NaN   
1        0.0        0.0  211.242170         8.106  ...           NaN   
2        0.0        0.0  211.289143         8.106  ...           NaN   
3        0.0        0.0  211.319643         8.106  .

In [ ]:
!pip install prophet 

In [4]:
from prophet import Prophet
import pandas as pd

# Dummy data
df_dummy = pd.DataFrame({
    'ds': ['2010-01-01', '2010-01-08', '2010-01-15'],
    'y':  [1, 2, 3]
})

# Dummy holidays
holidays_dummy = pd.DataFrame([
    {'holiday': 'test', 'ds': '2010-01-01'}
])

try:
    m_test = Prophet(holidays=holidays_dummy)
    m_test.fit(df_dummy)
    print("✅ Prophet works fine!")
except Exception as e:
    print(f"❌ Prophet Installation Error: {e}")


Importing plotly failed. Interactive plots will not work.
20:28:20 - cmdstanpy - INFO - Chain [1] start processing
20:28:20 - cmdstanpy - INFO - Chain [1] done processing


✅ Prophet works fine!


In [1]:
from prophet import Prophet
import pandas as pd
import numpy as np
import wandb
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. INITIALIZE WANDB RUN
# ---------------------------------------------------------
run = wandb.init(
    project="walmart-sales-forecasting",
    group="Prophet_Architecture", 
    name="Prophet_Val_Evaluation1",
    config={
        "daily_seasonality": False,
        "weekly_seasonality": True,
        "yearly_seasonality": True,
        "interval_width": 0.80
    }
)

# ---------------------------------------------------------
# 2. DEFINE HOLIDAYS (Strict Strings)
# ---------------------------------------------------------
# Create a list of dicts, then convert to DataFrame
holiday_list = []
for h_name, dates in [
    ('super_bowl', ['2010-02-14', '2011-02-06', '2012-02-05']),
    ('labor_day', ['2010-09-06', '2011-09-05', '2012-09-03']),
    ('thanksgiving', ['2010-11-25', '2011-11-24', '2012-11-22']),
    ('christmas', ['2010-12-25', '2011-12-25', '2012-12-25'])
]:
    for d in dates:
        holiday_list.append({'holiday': str(h_name), 'ds': str(d)})

holidays = pd.DataFrame(holiday_list)
holidays['lower_window'] = -1
holidays['upper_window'] = 1

# Verify Holidays DF
assert holidays['ds'].dtype == object, "Holiday ds must be object/string"
print("Holidays OK")

# ---------------------------------------------------------
# 3. PREPARE DATA (Strict Strings)
# ---------------------------------------------------------
VAL_START_DATE_STR = '2012-09-01'

# Convert Date to String YYYY-MM-DD
if 'Date' in df_train_full.columns:
    df_train_full['Date'] = pd.to_datetime(df_train_full['Date']).dt.strftime('%Y-%m-%d')

df_prophet_train = df_train_full[df_train_full['Date'] < VAL_START_DATE_STR].copy()
df_prophet_val = df_train_full[df_train_full['Date'] >= VAL_START_DATE_STR].copy()

store_depts = df_prophet_train[['Store', 'Dept']].drop_duplicates()
total_pairs = len(store_depts)
print(f"Total Store-Dept pairs to process: {total_pairs}")

predictions_list = []
processed_count = 0

# ---------------------------------------------------------
# 4. LOOP
# ---------------------------------------------------------
for idx, row in store_depts.iterrows():
    store_id = int(row['Store']) # Ensure int
    dept_id = int(row['Dept'])   # Ensure int
    
    # --- A. Prepare Historical Training Data ---
    df_hist = df_prophet_train[
        (df_prophet_train['Store'] == store_id) & 
        (df_prophet_train['Dept'] == dept_id)
    ].copy()
    
    # Rename columns
    df_prophet_data = df_hist.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    df_prophet_data = df_prophet_data.dropna(subset=['y'])
    
    if len(df_prophet_data) < 5:
        continue
        
    # CRITICAL: Reset index, convert ds to string, y to float
    df_prophet_data.reset_index(drop=True, inplace=True)
    df_prophet_data['ds'] = df_prophet_data['ds'].astype(str).str.strip()
    df_prophet_data['y'] = df_prophet_data['y'].astype(float)
        
    # --- B. Prepare Validation Dates ---
    val_dates_series = df_prophet_val[
        (df_prophet_val['Store'] == store_id) & 
        (df_prophet_val['Dept'] == dept_id)
    ]['Date']
    
    if len(val_dates_series) == 0:
        continue
    
    # CRITICAL: Clean string dates
    val_dates_list = val_dates_series.astype(str).str.strip().tolist()
    
    # Create Future DataFrame
    future_df = pd.DataFrame({'ds': val_dates_list})
    
    # --- C. Fit and Predict ---
    try:
        # Initialize Model
        m = Prophet(
            holidays=holidays, 
            daily_seasonality=False, 
            weekly_seasonality=True, 
            yearly_seasonality=True,
            interval_width=0.80
        )
        
        # Fit Model
        m.fit(df_prophet_data)
        
        # Predict
        forecast = m.predict(future_df)
        
        preds = forecast[['ds', 'yhat']]
        preds['yhat'] = preds['yhat'].clip(lower=0)
        
        # Add metadata
        preds['Store'] = store_id
        preds['Dept'] = dept_id
        
        predictions_list.append(preds)
        processed_count += 1
        
        if processed_count % 100 == 0:
            print(f"... Processed {processed_count}/{total_pairs} models")
            
    except Exception as e:
        if processed_count == 0:
            print(f"First Error Details: {e}")
            print(f"Type of ds in train: {type(df_prophet_data['ds'].iloc)}")
            print(f"Value of ds in train: {df_prophet_data['ds'].iloc}")
        pass

print(f"Successfully processed {processed_count} models.")

# ---------------------------------------------------------
# 5. EVALUATE RESULTS
# ---------------------------------------------------------
if not predictions_list:
    print("ERROR: No predictions generated.")
else:
    all_preds = pd.concat(predictions_list, ignore_index=True)
    
    # FIX: Convert Prophet's output 'ds' (datetime) to string to match validation set
    all_preds['ds'] = all_preds['ds'].dt.strftime('%Y-%m-%d')
    
    # Merge with actual validation data
    df_prophet_val_merge = df_prophet_val.copy()
    # Ensure validation Date is also string
    df_prophet_val_merge['Date'] = df_prophet_val_merge['Date'].astype(str).str.strip()
    
    merged_eval = all_preds.merge(
        df_prophet_val_merge[['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']], 
        left_on=['Store', 'Dept', 'ds'], 
        right_on=['Store', 'Dept', 'Date'], 
        how='inner'
    )
    
    if not merged_eval.empty:
        y_true = merged_eval['Weekly_Sales']
        y_pred = merged_eval['yhat']
        
        weights = np.where(merged_eval['IsHoliday'], 5, 1)
        
        val_wmae = np.average(np.abs(y_true - y_pred), weights=weights)
        val_mae = np.mean(np.abs(y_true - y_pred))
        
        print(f"Validation MAE: {val_mae:.2f}")
        print(f"Validation WMAE: {val_wmae:.2f}")
        
        wandb.log({"validation_mae": val_mae, "validation_wmae": val_wmae})
    else:
        print("No overlap between predictions and validation set after merge.")

wandb.finish()



Importing plotly failed. Interactive plots will not work.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/sandro/.netrc.
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Holidays OK


NameError: name 'df_train_full' is not defined